# Notebook de test du model de chatbot médical

In [ ]:
from huggingface_hub import InferenceClient
import os
import json
from datetime import datetime

# Import de la clef d'API (via variable d'environnement) et du modèle
api_key = os.getenv("HF_API_KEY")
modelID="meta-llama/Llama-3.2-3B-Instruct"

Création de la classe de discussion

In [79]:
class Discussion ():
    def __init__(self, modelID, api_key, window=5, max_token=500, temp=0.2):
        self.client = InferenceClient(api_key=api_key)
        self.modelID = modelID
        self.window = window #longueur de la fenêtre glissante de mémoire
        self.max_token = max_token #longueur de la réponse de l'IA
        self.temp = temp
        self.hist = []
        self.intent = None
        
        #Définition des prompts système en fonction du type de discussion
        self.prompt_sys = {}

    def save_log(self, filename="./logs/log_discussion"):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{filename}_{timestamp}.json"
        
        data_to_save = {
            "metadata": {
                "intent": self.intent,
                "timestamp": timestamp
            },
            "history": self.hist
        }

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, ensure_ascii=False, indent=4)
        print(f"Historique sauvegardé avec succès dans : {filename}")
    
    def load_syst_prompt(self, filename):
        prompts = {}
        current_key = None
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith('[') and line.endswith(']'):
                    current_key = line[1:-1]
                    prompts[current_key] = ""
                elif current_key and line:
                    prompts[current_key] += line + " "
        self.prompt_sys=prompts

    def call_AI(self, context):
        answer_gen = self.client.chat.completions.create(
                model=self.modelID,
                messages=context,
                max_tokens=self.max_token,
                temperature=self.temp)
        return answer_gen.choices[0].message.content
    
    def process_input(self, user_input):
        self.hist.append({"role":"user","content":user_input}) #Ajout de l'entrée de l'utilisateur à l'historique
        
        #Test de l'intention si elle n'a pas encore été détectée
        if self.intent is None : 
            print("% Détection de l'intention")
            context_detection = [{"role":"system","content":self.prompt_sys["INTENT"]}] + self.hist[-1:]
            intent = self.call_AI(context_detection).upper()

            if "MEDICAL" in intent: self.intent = "MEDICAL"
            elif "ADMIN" in intent: self.intent = "ADMIN"
            else: self.intent = "AUTRE"
            print("%",self.intent)
        
        #Réponse à la requete en fonction de l'intention
        if self.intent == "AUTRE":
            answer_fin= self.prompt_sys["AUTRE"]
            self.intent = None
        else :
            #Créer la fenêtre de contexte :
            context = [{"role":"system","content":self.prompt_sys[self.intent]}] + self.hist[-self.window:]
            
            #Call l'IA pour répondre :
            answer_fin = self.call_AI(context)
            
        self.hist.append({"role":"assistant","content":answer_fin})

        return answer_fin


Fonction de mise en page

In [77]:
def print_chat(chat, source, word_width=12):
    for paragraph in chat.split('\n'):
        paragraph = paragraph.split()
        margin = ""
        if source == "user":
            margin = "                               "
        for i in range(0,len(paragraph),word_width):
            print(margin, *paragraph[i:i+word_width])
        print()

Pipeline principale

In [ ]:
chat = Discussion(modelID=modelID,api_key=api_key)
filename = "prompt_syst.txt"
chat.load_syst_prompt(filename)

In [81]:
print("Bonjour, je suis un assistant spécialisé dans les questions médicales. Quelle est votre requette ?")
print(" Je vous invite à être le plus précis possible afin que je puisse vous répondre correctement.")
print("Pour terminer la discussion à tout moment tapez 'FIN'.")

while True:
    user_input= input("Vous : ")
    print_chat(user_input,"user")

    if user_input.upper() == "FIN":
        print("\n--- Fin de la simulation ---")
        break
    
    answer = chat.process_input(user_input=user_input)
    print_chat(answer, "IA")

Bonjour, je suis un assistant spécialisé dans les questions médicales. Quelle est votre requette ?
 Je vous invite à être le plus précis possible afin que je puisse vous répondre correctement.
Pour terminer la discussion à tout moment tapez 'FIN'.
                                J'ai mal à la tête, est ce que je dois téléconsulter un
                                médecin ?

% Détection de l'intention
% MEDICAL
 Si vous avez mal à la tête, il est recommandé de consulter
 un médecin pour évaluer la cause de vos symptômes et recevoir un
 diagnostic et un traitement appropriés.

                                d'accord

 Si vous avez mal à la tête, je vous conseille de consulter
 un médecin au plus vite ou de solliciter une téléconsultation sur une
 borne Tessan. Vous pouvez également appeler le 15 en cas d'urgence vitale.

                                fin


--- Fin de la simulation ---


In [ ]:
chat.save_log()